# SplatAD / OmniRE trên PandaSet

Train **R0 rigid**, **R1 SMPL cố định** và **R2 SMPL tối ưu pose** trên GPU Colab. Notebook xuất checkpoint, probe RGB/LiDAR, metric trên quan sát gốc và video toàn cảnh 60 FPS.

Chọn **Runtime → Change runtime type → GPU**. Đặt tệp SMPL Neutral v1.1 từ trang SMPL chính thức tại `MyDrive/SplatAD-OmniRE/SMPL_NEUTRAL.pkl`. Nếu chạy từ bản ZIP của repo, đặt `SplatAD-OmniRE.zip` cùng thư mục trên Drive.

- `smoke`: cảnh `011/front_right`, 200 bước mỗi cấu hình.
- `pilot`: ba cảnh, 1.200 bước mỗi cấu hình. Dùng tệp review khi đã xem các crop, nhất là nhóm xa.
- `confirm`: 4.000 bước hoặc tiếp tục pilot bằng cùng mã run. Các mốc train dùng chung lịch learning rate và độ phân giải.

Chuẩn bị pose từ bộ OmniRe đã phát hành hoặc dùng cell tiền xử lý riêng. Giữ checkpoint và kết quả trên Drive để resume qua các phiên. [Tài liệu PandaSet](https://huggingface.co/datasets/georghess/pandaset) · [SMPL](https://smpl.is.tue.mpg.de/) · [Pose OmniRe](https://github.com/ziyc/drivestudio/blob/e59bda4fa681f829dbb1d65f0de582b0f633c450/docs/HumanPose.md)

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

from google.colab import drive, output

drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive/SplatAD-OmniRE")
PERSIST.mkdir(exist_ok=True)
REPO = Path("/content/SplatAD-OmniRE")
GITHUB = "https://github.com/buide03/SplatAD-OmniRE.git"
ARCHIVE = PERSIST / "SplatAD-OmniRE.zip"

def shell(*args, cwd=None):
    subprocess.run([str(a) for a in args], cwd=cwd, check=True)

shell("nvidia-smi")
if not (REPO / "versions.json").exists():
    if ARCHIVE.exists():
        REPO.mkdir(exist_ok=True)
        shutil.unpack_archive(ARCHIVE, REPO)
    else:
        shell("git", "clone", GITHUB, REPO)
assert (REPO / "versions.json").exists(), "Thiếu mã nguồn repo ở /content/SplatAD-OmniRE."

STAGE = "smoke"  # smoke | pilot | confirm
RUN_ID = STAGE
STEPS = {"smoke": 200, "pilot": 1200, "confirm": 4000}[STAGE]
PILOT = json.loads((REPO / "protocol/pilot.json").read_text(encoding="utf-8"))
SCENES = ["011"] if STAGE == "smoke" else [entry["scene_id"] for entry in PILOT["scenes"]]
CAMERAS = ["front_right"] if STAGE == "smoke" else [
    "front_right", "front", "front_left", "right", "left", "back"
]
USE_PUBLIC_POSES = True
SMPL_PICKLE = PERSIST / "SMPL_NEUTRAL.pkl"
MODEL_VERSION = "smpl_neutral_v1.1"
RAW = Path("/content/pandaset")
PROCESSED = PERSIST / "omnire-pose"
INPUTS = PERSIST / "inputs"
RUNS = PERSIST / "runs"
ARTIFACTS = PERSIST / "artifacts"
PREP_TAG = "smoke" if STAGE == "smoke" else "pilot"
INPUTS.mkdir(exist_ok=True)
for folder in (RUNS, ARTIFACTS):
    folder.mkdir(exist_ok=True)
BODY = INPUTS / f"{MODEL_VERSION}.npz"
assert SMPL_PICKLE.is_file(), f"Đặt mô hình Neutral hợp lệ tại {SMPL_PICKLE}"

## Môi trường train và dữ liệu gốc

Bootstrap dùng Python 3.12, PyTorch CUDA 12.8 hoặc 12.6 và fork GSplat được pin theo commit. Lần gọi renderer đầu tiên sẽ biên dịch kernel. Script tải riêng các cảnh đã chọn trong ZIP PandaSet.

In [ ]:
shell("bash", "scripts/bootstrap-colab.sh", "train", cwd=REPO)
ACTIVATE = REPO / ".deps/train-env/activate-research"

def research(module, *args):
    shell("bash", "-c", 'source "$1"; shift; exec python -m "$@"',
          "splatad", ACTIVATE, f"research.{module}", *args, cwd=REPO)

research("download", "--scenes", *SCENES, "--output", RAW)
if shutil.which("ffmpeg") is None:
    shell("apt-get", "-qq", "update")
    shell("apt-get", "-qq", "install", "-y", "ffmpeg")

## Pose khởi tạo và manifest

`USE_PUBLIC_POSES=True` nạp pose công khai của OmniRe. Chuyển thành `False` để chạy 4D-Humans trong môi trường Python 3.10 riêng. Dùng `inputs/SCENE-review.json` để đánh dấu độ che khuất, tay/chân nhìn rõ và keypoint; cấu trúc có trong README. Tệp manifest cố định được dùng lại khi resume và khi so sánh.

In [ ]:
if not BODY.exists():
    SMPL_ENV = REPO / ".deps/smpl-convert"
    shell("uv", "venv", "--allow-existing", "--python", "3.10", SMPL_ENV, cwd=REPO)
    smpl_python = SMPL_ENV / "bin/python"
    shell("uv", "pip", "install", "--python", smpl_python, "setuptools<70", "wheel", "pip", cwd=REPO)
    shell("uv", "pip", "install", "--python", smpl_python, "--no-build-isolation",
          "numpy==1.26.4", "scipy==1.12.0", "chumpy==0.70", cwd=REPO)
    shell(smpl_python, "-m", "research.prepare", "prepare-smpl",
          "--input", SMPL_PICKLE, "--output", BODY, "--model-version", MODEL_VERSION, cwd=REPO)

if USE_PUBLIC_POSES:
    research("omnire", "--raw", RAW, "--processed", PROCESSED,
             "--archive", INPUTS / "pandaset_processed_humanpose.zip", "--scenes", *SCENES)
else:
    shell("bash", "scripts/bootstrap-colab.sh", "pose", cwd=REPO)
    for scene in SCENES:
        if not (PROCESSED / scene / "humanpose/smpl.pkl").exists():
            shell("bash", "scripts/omnire-pose.sh", RAW, PROCESSED, scene, SMPL_PICKLE, cwd=REPO)

def tracks_for(scene):
    return INPUTS / f"{scene}-{PREP_TAG}-tracks.json"

def manifest_for(scene):
    return INPUTS / f"{scene}-{PREP_TAG}-manifest.json"

for scene in SCENES:
    tracks, manifest = tracks_for(scene), manifest_for(scene)
    if not tracks.exists():
        research("prepare", "export-tracks", "--raw", RAW, "--processed", PROCESSED,
                 "--body", BODY, "--scene", scene, "--output", tracks)
    if not manifest.exists():
        review = INPUTS / f"{scene}-review.json"
        if not review.is_file():
            review = REPO / "protocol" / f"{scene}-review.json"
        research("prepare", "inventory", "--raw", RAW, "--tracks", tracks, "--scene", scene,
                 "--cameras", *CAMERAS, "--output", manifest,
                 *(["--review", review] if review.is_file() else []))
    spec = json.loads(manifest.read_text())
    print(scene, len(spec["selected_uuids"]), "UUID định tuyến SMPL")
    assert spec["selected_uuids"], f"Cần track SMPL hợp lệ cho cảnh {scene}"
    if STAGE != "smoke":
        common = {}
        for row in spec["observations"]:
            if row["usable"] and row["smpl_valid"] and row["uuid"] in spec["selected_uuids"]:
                common.setdefault((row["camera"], row["frame"]), set()).add(row["uuid"])
        assert max(map(len, common.values()), default=0) >= 2, f"Cảnh {scene} cần hai UUID SMPL cùng xuất hiện"


## Train đối chứng và kiểm tra gradient

R0 chạy trước để kiểm tra RGB và LiDAR trên renderer gốc. R1 và R2 chia sẻ khởi tạo Gaussian người. Probe R2 kiểm tra gradient và thay đổi tham số sau Adam theo từng cảm biến. Cell tự tiếp tục run dang dở từ checkpoint mới nhất.

In [ ]:
def run_folder(scene, variant):
    return RUNS / f"{scene}-{variant}" / "splatad-smpl" / RUN_ID

def checkpoint_for(scene, variant):
    folder = run_folder(scene, variant)
    return max((folder / "nerfstudio_models").glob("step-*.ckpt"), default=None)

RESULTS = ARTIFACTS / RUN_ID
RESULTS.mkdir(exist_ok=True)
for scene in SCENES:
    for variant in ("R0", "R1", "R2"):
        folder = run_folder(scene, variant)
        config = folder / "config.yml"
        checkpoint = checkpoint_for(scene, variant)
        done = int(checkpoint.stem.rsplit("-", 1)[1]) + 1 if checkpoint else 0
        if done < STEPS:
            if checkpoint:
                research("train", "--resume", config, "--checkpoint", checkpoint, "--steps", STEPS - done)
            else:
                research("train", "--raw", RAW, "--scene", scene, "--variant", variant,
                         "--tracks", tracks_for(scene), "--body", BODY, "--manifest", manifest_for(scene),
                         "--cameras", *CAMERAS, "--output", RUNS, "--run-id", RUN_ID, "--steps", STEPS,
                         "--checkpoint-every", 200 if STAGE == "smoke" else 500,
                         "--monitor-every", 200 if STAGE == "smoke" else 500)
        if variant in ("R0", "R2"):
            checkpoint = checkpoint_for(scene, variant)
            research("probe", "--config", config, "--checkpoint", checkpoint, "--manifest", manifest_for(scene),
                     "--output", RESULTS / f"{scene}-{variant}-probe.json")

## Đánh giá và render toàn cảnh

Metric dùng ảnh và tia tại mốc cảm biến gốc. Video dùng render mới tại từng mốc nội suy. `render_fps` là tốc độ dựng trên GPU và `fps` là tốc độ phát của MP4. So sánh bù chuyển động từ cùng checkpoint R2 bằng cách đặt riêng vận tốc Gaussian người về 0.

In [ ]:
REPLAY_FPS = 60
REPLAY_SCALE = 1.0
PROBES_ONLY = STAGE != "confirm"
folders = []
for scene in SCENES:
    for variant in ("R0", "R1", "R2"):
        config = run_folder(scene, variant) / "config.yml"
        checkpoint = checkpoint_for(scene, variant)
        for zero_velocity in (False, True) if variant == "R2" else (False,):
            suffix = "-velocity-off" if zero_velocity else ""
            folder = RESULTS / f"{scene}-{variant}{suffix}-{checkpoint.stem}"
            flags = ["--zero-smpl-velocity"] if zero_velocity else []
            shared = ["--config", config, "--checkpoint", checkpoint,
                      "--manifest", manifest_for(scene), "--output", folder, *flags]
            research("evaluate", *shared, *(["--probes"] if PROBES_ONLY else []))
            research("replay", *shared, "--fps", REPLAY_FPS, "--scale", REPLAY_SCALE)
            folders.append(folder)
research("compare", "--runs", *folders, "--output", RESULTS / "comparison")

## Dashboard · Video, metric và kiểm tra khung hình

Chọn cấu hình, góc camera, UUID và nhóm cự ly. Xem loss và gradient theo cảm biến. Tệp `comparison.csv` chứa tổng hợp theo cặp trên các quan sát chung.

In [ ]:
if "dashboard" in globals() and dashboard.poll() is None:
    dashboard.terminate()
    dashboard.wait(timeout=5)
dashboard = subprocess.Popen([sys.executable, "-m", "research.dashboard",
                              "--artifacts", str(ARTIFACTS), "--port", "8765"], cwd=REPO)
time.sleep(1)
assert dashboard.poll() is None, "Không khởi động được dashboard."
output.serve_kernel_port_as_iframe(8765, height=1000)